In [5]:
%load_ext autoreload
%autoreload 2

import numpy as np

from liquidation_task_tools.constants import SECOND, TWO_WEEKS_SEC, BYBIT_LIQUIDATION_DELAY_US
from liquidation_task_tools.loaders import ParquetDataLoader
from liquidation_task_tools.labeling import calculate_model_pnl
from liquidation_task_tools.features import (
    LiqudationClusterImbalance,
    LiqudationClusterStrength,
    LiqudationClusterTotalNotional,
    LiqudationClusterCount,
    TradeSideLiquidationImbalance,
    TradeSideLiquidationStrength,
    BboSpreadBps,
    BboVolumeImbalance,
    BboVolumeImbalanceAbs,
    BboMidSmoothDeltaBps,
    BboTopDepthLog,
    BboMidDeltaBps,
    BboMicroPricePremiumBps,
    TradeBboEdgeBps,
    TradeSideBboVolumeImbalance,
    TradeSideBboMicroPricePremiumBps,
    TradeSideBboMidDeltaBps,
    TradeSideBboMidSmoothDeltaBps,
    TradeSide,
    TradeNotionalLog,
    TradeSignedNotionalLog,
    TradeFlowImbalance,
    TradeSideFlowImbalance,
    TradeFlowToxicity,
    TradeFlowNotionalLog,
    TradeFlowCountLog,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from pathlib import Path

binance_btc_bbo = "binance_booktickers_btc"
binance_btc_liq = "binance_liquidations_btc"
binance_btc_trades = "binance_trades_btc"
bybit_btc_liq = "bybit_liquidations_btc"

parquet_names = [
    binance_btc_bbo,
    binance_btc_liq,
    binance_btc_trades,
    bybit_btc_liq,
]

def _resolve_data_root() -> Path:
    candidates = (
        Path.cwd() / "liquidation_task_tools" / "data",
        Path.cwd() / "liquidation_task" / "data",
        Path.cwd() / "data",
        Path.cwd().parent / "liquidation_task_tools" / "data",
        Path.cwd().parent / "liquidation_task" / "data",
        Path.cwd().parent / "data",
    )
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find data directory in known locations")


data_root = _resolve_data_root()
parquet_paths = [
    str(data_root / "binance_booktickers" / "perp_btcusdt.parquet"),
    str(data_root / "binance_liquidations" / "perp_btcusdt.parquet"),
    str(data_root / "binance_trades" / "perp_btcusdt.parquet"),
    str(data_root / "bybit_liquidations" / "btcusdt.parquet"),
]


## Features


In [7]:
from liquidation_task_tools.training import FeatureSpec

window_sec = 30 * SECOND
horizons_sec = (30, 120, 300)
tau_index = 2


def build_sample_weights(chunk):
    trades_df = chunk[binance_btc_trades]
    notionals = (trades_df["price"] * trades_df["amount"]).to_numpy().astype(np.float64)
    return np.minimum(notionals, 100_000.0)


feature_specs = [
    FeatureSpec(
        feature=LiqudationClusterImbalance(),
        source_map={"liquidations": binance_btc_liq, "trades": binance_btc_trades},
        params={"window_sec": window_sec},
    ),
    FeatureSpec(
        feature=LiqudationClusterStrength(),
        source_map={"liquidations": binance_btc_liq, "trades": binance_btc_trades},
        params={"window_sec": window_sec},
    ),
    FeatureSpec(
        feature=LiqudationClusterTotalNotional(),
        source_map={"liquidations": binance_btc_liq, "trades": binance_btc_trades},
        params={"window_sec": window_sec},
    ),
    FeatureSpec(
        feature=LiqudationClusterCount(),
        source_map={"liquidations": binance_btc_liq, "trades": binance_btc_trades},
        params={"window_sec": window_sec},
    ),
    FeatureSpec(
        feature=TradeSideLiquidationImbalance(),
        source_map={"liquidations": binance_btc_liq, "trades": binance_btc_trades},
        params={"window_sec": window_sec},
    ),
    FeatureSpec(
        feature=TradeSideLiquidationStrength(),
        source_map={"liquidations": binance_btc_liq, "trades": binance_btc_trades},
        params={"window_sec": window_sec},
    ),
    
    FeatureSpec(
        feature=BboSpreadBps(),
        source_map={"bbo": binance_btc_bbo, "trades": binance_btc_trades},
        params={"window_sec": window_sec},
    ),
    FeatureSpec(
        feature=BboVolumeImbalance(),
        source_map={"bbo": binance_btc_bbo, "trades": binance_btc_trades},
        params={"window_sec": window_sec},
    ),
    FeatureSpec(
        feature=BboMidSmoothDeltaBps(),
        source_map={"bbo": binance_btc_bbo, "trades": binance_btc_trades},
        params={"window_sec": window_sec},
    ),
    FeatureSpec(
        feature=BboTopDepthLog(),
        source_map={"bbo": binance_btc_bbo, "trades": binance_btc_trades},
        params={"window_sec": window_sec},
    ),

    # FeatureSpec(
    #     feature=TradeSideBboVolumeImbalance(),
    #     source_map={"bbo": binance_btc_bbo, "trades": binance_btc_trades},
    #     params={"window_sec": window_sec},
    # ),
    FeatureSpec(
        feature=TradeSideBboMicroPricePremiumBps(),
        source_map={"bbo": binance_btc_bbo, "trades": binance_btc_trades},
        params={"window_sec": window_sec},
    ),
    # ),
    # FeatureSpec(
    #     feature=TradeSideBboMidDeltaBps(),
    #     source_map={"bbo": binance_btc_bbo, "trades": binance_btc_trades},
    #     params={"window_sec": window_sec},
    # ),
    # FeatureSpec(
    #     feature=TradeSideBboMidSmoothDeltaBps(),
    #     source_map={"bbo": binance_btc_bbo, "trades": binance_btc_trades},
    #     params={"window_sec": window_sec},
    # ),
    # FeatureSpec(
    #     feature=TradeSignedNotionalLog(),
    #     source_map={"trades": binance_btc_trades},
    # ),
    FeatureSpec(
        feature=TradeFlowImbalance(),
        source_map={"trades": binance_btc_trades},
        params={"window_sec": window_sec},
    ),
    # FeatureSpec(
    #     feature=TradeFlowToxicity(),
    #     source_map={"trades": binance_btc_trades},
    #     params={"window_sec": window_sec},
    # ),
]

In [ ]:
import polars as pl
from datetime import datetime, timezone

from liquidation_task_tools.labeling import calculate_model_pnl
from liquidation_task_tools.training import FeatureSpec, RankingPipeline, build_model

TRAIN_UNTIL_TS_SEC = int(datetime(2026, 2, 1, tzinfo=timezone.utc).timestamp())
TARGET_MAX_HORIZON_SEC = 300

VALID_FROM_TS_SEC = int(datetime(2026, 2, 1, tzinfo=timezone.utc).timestamp())
VALID_TO_TS_SEC = int(datetime(2026, 3, 1, tzinfo=timezone.utc).timestamp())

RANK_TRAIN_MAX_DAYS = 1
RANK_TRAIN_FROM_TS_SEC = max(0, TRAIN_UNTIL_TS_SEC - RANK_TRAIN_MAX_DAYS * 24 * 60 * 60)

RANK_TRAIN_CHUNK_SEC = 60 * 60
RANK_VAL_CHUNK_SEC = 60 * 60
RANK_VAL_MAX_DAYS = 1
VALID_UNTIL_TS_SEC = min(VALID_TO_TS_SEC, VALID_FROM_TS_SEC + RANK_VAL_MAX_DAYS * 24 * 60 * 60)

RANK_GROUP_SEC = 1

datafiles = [
    ParquetDataLoader.Datafile(name, path)
    for name, path in zip(parquet_names, parquet_paths)
]


def build_rank_target(chunk):
    stats = calculate_model_pnl(
        chunk[binance_btc_trades],
        chunk[binance_btc_bbo],
        horizons_sec=horizons_sec,
    )
    target = (-stats["trade_pnl"][:, tau_index]).astype(np.float32)
    valid_mask = np.asarray(stats["valid_mask"][:, tau_index], dtype=bool)
    target = np.where(valid_mask, target, 0.0)
    return np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)


def build_rank_weights(chunk):
    trades = chunk[binance_btc_trades]
    notionals = (trades["price"] * trades["amount"]).to_numpy().astype(np.float32)
    return np.minimum(notionals, 100_000.0)


def build_rank_group_id(chunk):
    ts = chunk[binance_btc_trades]["timestamp"].to_numpy().astype(np.int64)
    return (ts // (RANK_GROUP_SEC * SECOND)).astype(np.int64)


ranking_feature_specs = feature_specs + [
    FeatureSpec(
        feature=LiqudationClusterImbalance(name="bybit_liq_imbalance"),
        source_map={"liquidations": bybit_btc_liq, "trades": binance_btc_trades},
        params={"window_sec": window_sec, "timestamp_shift_us": BYBIT_LIQUIDATION_DELAY_US},
    ),
    FeatureSpec(
        feature=TradeSideLiquidationStrength(name="trade_side_bybit_liq_strength"),
        source_map={"liquidations": bybit_btc_liq, "trades": binance_btc_trades},
        params={"window_sec": window_sec, "timestamp_shift_us": BYBIT_LIQUIDATION_DELAY_US},
    ),
]

common_loader_kwargs = dict(
    lookback_by_name={
        binance_btc_bbo: 31,
        binance_btc_liq: 31,
        bybit_btc_liq: 31,
    },
    lookahead_by_name={binance_btc_bbo: TARGET_MAX_HORIZON_SEC},
)

rank_train_loader = ParquetDataLoader(
    datafiles,
    RANK_TRAIN_CHUNK_SEC,
    ParquetDataLoader.InputTimeScale.sec,
    since=RANK_TRAIN_FROM_TS_SEC,
    until=TRAIN_UNTIL_TS_SEC,
    **common_loader_kwargs,
)
rank_val_loader = ParquetDataLoader(
    datafiles,
    RANK_VAL_CHUNK_SEC,
    ParquetDataLoader.InputTimeScale.sec,
    since=VALID_FROM_TS_SEC,
    until=VALID_UNTIL_TS_SEC,
    **common_loader_kwargs,
)

rank_model = build_model(
    "catboost_chunk",
    "ranking",
    {
        "loss_function": "YetiRank",
        "iterations": 40,
        "depth": 3,
        "learning_rate": 0.03,
        "random_seed": 42,
        "allow_writing_files": False,
        "thread_count": 1,
        "verbose": 25,
    },
)

ranking_pipeline = RankingPipeline(
    model=rank_model,
    feature_specs=ranking_feature_specs,
    data_loader=rank_train_loader,
    target_builder=build_rank_target,
    sample_weight_builder=None,
    group_id_builder=build_rank_group_id,
)
ranking_pipeline.fit()


0:	total: 207ms	remaining: 8.09s
25:	total: 5.29s	remaining: 2.85s
39:	total: 8.19s	remaining: 0us
0:	total: 163ms	remaining: 6.34s
25:	total: 4.18s	remaining: 2.25s
39:	total: 6.42s	remaining: 0us
0:	total: 162ms	remaining: 6.33s
25:	total: 4.12s	remaining: 2.22s
39:	total: 6.33s	remaining: 0us
0:	total: 66ms	remaining: 2.57s
25:	total: 1.69s	remaining: 909ms
39:	total: 2.58s	remaining: 0us
0:	total: 72.6ms	remaining: 2.83s
25:	total: 1.87s	remaining: 1.01s
39:	total: 2.87s	remaining: 0us
0:	total: 90.5ms	remaining: 3.53s
25:	total: 2.36s	remaining: 1.27s
39:	total: 3.64s	remaining: 0us
0:	total: 164ms	remaining: 6.38s
25:	total: 4.17s	remaining: 2.25s
39:	total: 6.42s	remaining: 0us
0:	total: 217ms	remaining: 8.46s
25:	total: 5.76s	remaining: 3.1s
39:	total: 8.86s	remaining: 0us
0:	total: 895ms	remaining: 34.9s
25:	total: 24.5s	remaining: 13.2s
39:	total: 37.9s	remaining: 0us
0:	total: 291ms	remaining: 11.3s
25:	total: 7.51s	remaining: 4.04s
39:	total: 11.6s	remaining: 0us
0:	total: 

In [ ]:
scores_parts = []
pnl_parts = []
valid_parts = []
weights_parts = []

rank_val_loader.reset()
for chunk in rank_val_loader:
    scores = np.asarray(ranking_pipeline.predict_chunk(chunk, proba=False), dtype=np.float32).reshape(-1)
    stats = calculate_model_pnl(
        chunk[binance_btc_trades],
        chunk[binance_btc_bbo],
        horizons_sec=horizons_sec,
    )
    scores_parts.append(scores)
    pnl_parts.append(np.asarray(stats["trade_pnl"][:, tau_index], dtype=np.float32))
    valid_parts.append(np.asarray(stats["valid_mask"][:, tau_index], dtype=bool))
    weights_parts.append(build_rank_weights(chunk).astype(np.float32, copy=False))
rank_val_loader.reset()

scores = np.concatenate(scores_parts)
pnl_tau = np.concatenate(pnl_parts)
valid_tau = np.concatenate(valid_parts)
weights = np.concatenate(weights_parts)

del scores_parts, pnl_parts, valid_parts, weights_parts

eligible = np.flatnonzero(valid_tau)
if eligible.size == 0:
    raise ValueError("No valid rows in validation window")

eligible_scores = scores[eligible]
eligible_pnl = pnl_tau[eligible]
eligible_weights = weights[eligible]

worst_first_idx = np.argsort(eligible_scores)[::-1]
worst_weights = eligible_weights[worst_first_idx]
worst_weighted_pnl = (eligible_pnl[worst_first_idx] * worst_weights).astype(np.float64)

cum_filtered_weight = np.concatenate(([0.0], np.cumsum(worst_weights, dtype=np.float64)))
cum_filtered_weighted_pnl = np.concatenate(([0.0], np.cumsum(worst_weighted_pnl, dtype=np.float64)))

total_weight = float(eligible_weights.sum(dtype=np.float64))
total_weighted_pnl = float((eligible_pnl * eligible_weights).sum(dtype=np.float64))
pnl_all = np.nan if total_weight == 0.0 else total_weighted_pnl / total_weight

eval_days = max(1.0, (VALID_UNTIL_TS_SEC - VALID_FROM_TS_SEC) / (24 * 60 * 60))

rows = []
for frac in np.linspace(0.0, 0.3, 31):
    k = int(eligible.size * frac)

    filtered_weight = float(cum_filtered_weight[k])
    filtered_weighted_pnl = float(cum_filtered_weighted_pnl[k])

    kept_weight = total_weight - filtered_weight
    kept_weighted_pnl = total_weighted_pnl - filtered_weighted_pnl

    pnl_kept = np.nan if kept_weight == 0.0 else kept_weighted_pnl / kept_weight
    pnl_filtered = np.nan if filtered_weight == 0.0 else filtered_weighted_pnl / filtered_weight

    rows.append(
        {
            "filter_fraction": float(frac),
            "score_bps": pnl_kept - pnl_all,
            "pnl_all_bps": pnl_all,
            "pnl_kept_bps": pnl_kept,
            "pnl_filtered_bps": pnl_filtered,
            "kept_turnover_per_day": kept_weight / eval_days,
        }
    )

val_result = pl.DataFrame(rows)
best = (
    val_result
    .filter(pl.col("kept_turnover_per_day") >= 500_000.0)
    .sort("score_bps", descending=True)
    .head(1)
)

display(best)
display(val_result.sort("score_bps", descending=True).head(10))

filter_fraction,score_bps,pnl_all_bps,pnl_kept_bps,pnl_filtered_bps,kept_turnover_per_day
f64,f64,f64,f64,f64,f64
0.21,0.933887,0.134271,1.068159,-2.677224,1.0175e10


filter_fraction,score_bps,pnl_all_bps,pnl_kept_bps,pnl_filtered_bps,kept_turnover_per_day
f64,f64,f64,f64,f64,f64
0.21,0.933887,0.134271,1.068159,-2.677224,1.0175e10
0.3,0.927955,0.134271,1.062226,-1.761256,9.0995e9
0.27,0.917854,0.134271,1.052126,-1.955962,9.4184e9
0.29,0.910588,0.134271,1.04486,-1.792615,9.2045e9
0.26,0.909762,0.134271,1.044034,-2.027174,9.5391e9
0.2,0.908341,0.134271,1.042612,-2.748278,1.0306e10
0.11,0.902846,0.134271,1.037117,-5.980854,1.1810e10
0.28,0.901884,0.134271,1.036155,-1.843343,9.3089e9
0.12,0.896307,0.134271,1.030578,-5.31232,1.1639e10


In [10]:
model_path = Path("artifacts/ranking_model.cbm")
model_path.parent.mkdir(parents=True, exist_ok=True)
ranking_pipeline.model._model.save_model(str(model_path))